# 🐍 Aula 17 — Automação de GPUs: versão enxuta (complemento da Aula 14)

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem
> 📌 **Complemento da [Aula 14](../../aula14/README.md).** Demo enxuta dos scripts no Colab (Linux)
> + **laboratório em Python** que faz o mesmo sem Bash, no Windows.

O servidor de treinamento da empresa ficou travado durante a madrugada — a GPU atingiu **95°C** e o job falhou silenciosamente. Ninguém percebeu até a manhã seguinte. O time precisa de um sistema de monitoramento automático **24h/7d** que colete métricas a cada 5 segundos, salve em CSV, gere alertas de temperatura e publique um dashboard diário no Google Sheets — tudo via scripts de automação (Bash no Colab; alternativa Python no laboratório).

In [ ]:
# 1. Verificar GPU disponível no Google Colab ou ambiente Linux
!nvidia-smi || echo "NVIDIA GPU não encontrada. Usando modo simulado para testes."

In [ ]:
%%writefile monitor_gpu.sh
# 2. Criar script monitor_gpu.sh para coleta de métricas em CSV
#!/usr/bin/env bash
set -euo pipefail

INTERVALO="${1:-5}"
SAIDA="${2:-gpu_log.csv}"
DURACAO="${3:-30}"

if [ "$INTERVALO" -le 0 ]; then INTERVALO=5; fi
MAX_AMOSTRAS=$(( DURACAO / INTERVALO ))

CABECALHO="timestamp,gpu_index,gpu_name,temp_c,util_gpu_pct,util_mem_pct,mem_used_mb,mem_total_mb,power_w,power_limit_w,clock_graphics_mhz,clock_mem_mhz"
echo "$CABECALHO" > "$SAIDA"

echo "Iniciando monitoramento -> $SAIDA ($MAX_AMOSTRAS amostras)"

AMOSTRA=0
while [ "$AMOSTRA" -lt "$MAX_AMOSTRAS" ]; do
    TS=$(date +"%Y-%m-%d %H:%M:%S")
    if command -v nvidia-smi &>/dev/null; then
        DADOS=$(nvidia-smi --query-gpu=index,name,temperature.gpu,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,power.limit,clocks.current.graphics,clocks.current.memory --format=csv,noheader,nounits)
    else
        DADOS="0, Tesla T4 (Simulada), $(( 45 + RANDOM % 40 )), $(( 20 + RANDOM % 75 )), $(( 10 + RANDOM % 60 )), $(( 2048 + RANDOM % 4000 )), 15360, $(( 45 + RANDOM % 80 )).0, 70.0, 1590, 5001"
    fi
    while IFS= read -r linha; do
        echo "$TS,$linha" >> "$SAIDA"
    done <<< "$DADOS"
    AMOSTRA=$(( AMOSTRA + 1 ))
    echo "  Amostra $AMOSTRA/$MAX_AMOSTRAS coletada."
    sleep "$INTERVALO"
done
echo "Coleta finalizada!"


In [ ]:
# Dar permissão e rodar monitor_gpu.sh por 20 segundos (4 amostras a cada 5s)
!chmod +x monitor_gpu.sh
!./monitor_gpu.sh 5 gpu_log.csv 20
!head -n 5 gpu_log.csv

In [ ]:
%%writefile alerta_gpu.sh
# 3. Criar o script alerta_gpu.sh
#!/usr/bin/env bash
set -euo pipefail

LIMITE_TEMP="${1:-75}"
LIMITE_UTIL="${2:-90}"
LOG_ALERTAS="gpu_alertas.log"

alerta() {
    local msg="$1"
    local ts
    ts=$(date +"%Y-%m-%d %H:%M:%S")
    echo "[$ts] ALERTA: $msg" | tee -a "$LOG_ALERTAS"
}

if command -v nvidia-smi &>/dev/null; then
    DADOS_ENTRADA=$(nvidia-smi --query-gpu=index,name,temperature.gpu,utilization.gpu --format=csv,noheader,nounits)
else
    DADOS_ENTRADA="0, Tesla T4 (Simulada), 82, 96"
fi

while IFS="," read -r idx nome temp util; do
    idx=$(echo "$idx" | xargs)
    nome=$(echo "$nome" | xargs)
    temp=$(echo "$temp" | xargs)
    util=$(echo "$util" | xargs)

    if [ "$temp" -ge "$LIMITE_TEMP" ]; then
        alerta "GPU $idx ($nome): temperatura ${temp}°C >= ${LIMITE_TEMP}°C"
    fi
    if [ "$util" -ge "$LIMITE_UTIL" ]; then
        alerta "GPU $idx ($nome): utilização ${util}% >= ${LIMITE_UTIL}%"
    fi
done <<< "$DADOS_ENTRADA"


In [ ]:
# Testar alerta_gpu.sh com limiar de 70°C
!chmod +x alerta_gpu.sh
!./alerta_gpu.sh 70 85
!cat gpu_alertas.log || true

In [ ]:
# 4. Simular agendamento em segundo plano via Python (simulador de cron)
import time, subprocess

print("Simulando execução do cron a cada 5 segundos (3 ciclos)...")
for i in range(1, 4):
    print(f"--- Ciclo {i} do Cron ---")
    subprocess.run(["./alerta_gpu.sh", "70", "85"])
    time.sleep(2)
print("Simulação de agendamento concluída.")

In [ ]:
# 5. Instalar gnuplot e gerar o dashboard visual (com fallback Matplotlib)
!apt-get install -y gnuplot > /dev/null 2>&1 || true

import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

df = pd.read_csv("gpu_log.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("GPU Monitoring Dashboard (Colab)", fontsize=14, fontweight='bold')

axes[0, 0].plot(df['timestamp'], df['temp_c'], color='#EF4444', marker='o', label='Temp C')
axes[0, 0].axhline(y=80, color='#F97316', linestyle='--', label='Limite 80C')
axes[0, 0].set_title("Temperatura (°C)")
axes[0, 0].legend()

axes[0, 1].plot(df['timestamp'], df['util_gpu_pct'], color='#10B981', marker='s', label='GPU Util %')
axes[0, 1].plot(df['timestamp'], df['util_mem_pct'], color='#6366F1', marker='^', label='Mem Util %')
axes[0, 1].set_title("Utilização (%)")
axes[0, 1].legend()

axes[1, 0].fill_between(df['timestamp'], df['mem_used_mb'], color='#7E22CE', alpha=0.3)
axes[1, 0].plot(df['timestamp'], df['mem_used_mb'], color='#7E22CE', label='VRAM Usada')
axes[1, 0].set_title("Memória VRAM (MB)")
axes[1, 0].legend()

axes[1, 1].plot(df['timestamp'], df['power_w'], color='#F97316', label='Power Draw')
axes[1, 1].plot(df['timestamp'], df['power_limit_w'], color='#EF4444', linestyle='--', label='Power Limit')
axes[1, 1].set_title("Potência (W)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("gpu_dashboard.png")
plt.close()

display(Image(filename="gpu_dashboard.png"))

In [ ]:
%%writefile enviar_para_sheets.py
# 6. Script de integração com Google Sheets API (enviar_para_sheets.py)
import os, csv, sys

csv_file = sys.argv[1] if len(sys.argv) > 1 else 'gpu_log.csv'
sheets_id = os.environ.get('SHEETS_ID', '')

if not sheets_id:
    print('[Modo Simulação] Variável SHEETS_ID não encontrada.')
    if os.path.exists(csv_file):
        with open(csv_file) as f:
            total = sum(1 for _ in f) - 1
        print(f'Pronto para enviar {total} registros de telemetria ao Google Sheets.')
else:
    print(f'Enviando métricas do arquivo {csv_file} para a planilha {sheets_id}...')


In [ ]:
!python3 enviar_para_sheets.py gpu_log.csv

---  
## 📝 7. Exercícios Práticos (5)  
Resolva os 5 exercícios a seguir completando as células de código correspondentes:

### Exercício 1: Alterar o Tempo de Coleta  
Modifique a execução do `monitor_gpu.sh` para coletar métricas a cada **2 segundos** durante um período de **10 segundos** e salve o resultado em `gpu_exercicio1.csv`.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# Exemplo: !./monitor_gpu.sh 2 gpu_exercicio1.csv 10

### Exercício 2: Criar Alerta para Memória VRAM  
Crie uma função Bash ou edite `alerta_gpu.sh` para emitir um alerta se o uso de memória alocada (`mem_used_mb`) for maior que **8000 MB**.

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# Crie o script alerta_vram.sh e teste a execução.

### Exercício 3: Adicionar a Métrica de Rotação do Cooler (`fan.speed`)  
Atualize a instrução `nvidia-smi --query-gpu` para incluir o campo `fan.speed` na coleta de métricas CSV.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# Adicione fan.speed na query e execute uma amostra.

### Exercício 4: Gráfico Comparativo Clock vs Temperatura  
Utilize `pandas` e `matplotlib` para gerar um gráfico de dois eixos Y (ou lado a lado) comparando `clock_graphics_mhz` e `temp_c` ao longo do tempo.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# Use pd.read_csv('gpu_log.csv') e gere a figura.

### Exercício 5: Simulação de Webhook Slack/Discord  
Escreva um trecho Python ou Bash que receba uma mensagem de alerta e faça uma requisição POST simulada (ou exiba o payload JSON que seria enviado).

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# Exemplo: import json, requests / curl -X POST ...

---  
## 📌 8. Síntese e Conclusão  

- O `nvidia-smi --query-gpu` é a base fundamental para extração limpa de telemetria em servidores NVIDIA.
- Scripts Bash com loops de amostragem permitem registrar o histórico em arquivos CSV para auditoria.
- Ferramentas de agendamento (`cron` / `systemd timers`) tornam a monitoria autônoma e resiliente 24h/7d.
- Dashboards visuais com `gnuplot` / `matplotlib` e integrações como Google Sheets garantem visibilidade remota da infraestrutura.